In [159]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/vaidehishetye/englishtohinditranslate/hindi to english dataset/opus.en-hi-test.en
/kaggle/input/datasets/vaidehishetye/englishtohinditranslate/hindi to english dataset/opus.en-hi-train.hi
/kaggle/input/datasets/vaidehishetye/englishtohinditranslate/hindi to english dataset/opus.en-hi-train.en
/kaggle/input/datasets/vaidehishetye/englishtohinditranslate/hindi to english dataset/opus.en-hi-dev.hi
/kaggle/input/datasets/vaidehishetye/englishtohinditranslate/hindi to english dataset/opus.en-hi-test.hi
/kaggle/input/datasets/vaidehishetye/englishtohinditranslate/hindi to english dataset/opus.en-hi-dev.en


In [160]:
import os

dataset_path = "/kaggle/input/datasets/vaidehishetye/englishtohinditranslate"

print(os.listdir(dataset_path))

['hindi to english dataset']


In [161]:
import os

dataset_path = "/kaggle/input/datasets/vaidehishetye/englishtohinditranslate/hindi to english dataset"

print(os.listdir(dataset_path))

['opus.en-hi-test.en', 'opus.en-hi-train.hi', 'opus.en-hi-train.en', 'opus.en-hi-dev.hi', 'opus.en-hi-test.hi', 'opus.en-hi-dev.en']


define paths

In [162]:
dataset_path = "/kaggle/input/datasets/vaidehishetye/englishtohinditranslate/hindi to english dataset"

train_en_path = dataset_path + "/opus.en-hi-train.en"
train_hi_path = dataset_path + "/opus.en-hi-train.hi"

dev_en_path = dataset_path + "/opus.en-hi-dev.en"
dev_hi_path = dataset_path + "/opus.en-hi-dev.hi"

test_en_path = dataset_path + "/opus.en-hi-test.en"
test_hi_path = dataset_path + "/opus.en-hi-test.hi"

load files

In [163]:
def load_data(path):

    with open(path, "r", encoding="utf-8") as f:
        lines = f.read().split("\n")

    return lines


train_en = load_data(train_en_path)
train_hi = load_data(train_hi_path)

dev_en = load_data(dev_en_path)
dev_hi = load_data(dev_hi_path)

test_en = load_data(test_en_path)
test_hi = load_data(test_hi_path)

check dataset

In [164]:
print(train_en[0])
print(train_hi[0])

Other, Private Use
अन्य, निज़ी उपयोग


check dataset size

In [165]:
print(len(train_en))
print(len(train_hi))

534320
534320


remove empty lines

In [166]:
clean_train_en = []
clean_train_hi = []

for en, hi in zip(train_en, train_hi):

    if en.strip() != "" and hi.strip() != "":

        clean_train_en.append(en.strip())
        clean_train_hi.append(hi.strip())

train_en = clean_train_en
train_hi = clean_train_hi

In [167]:
print(len(train_en))
print(len(train_hi))

534319
534319


observe sentences

In [168]:
for i in range(5):

    print("EN:", train_en[i])
    print("HI:", train_hi[i])
    print()

EN: Other, Private Use
HI: अन्य, निज़ी उपयोग

EN: [SCREAMING]
HI: ऊबड़ .

EN: Spouse
HI: जीवनसाथी

EN: I will never salute you!
HI: - तुम एक कमांडर कभी नहीं होगा!

EN: and the stars and the trees bow themselves;
HI: और तारे और वृक्ष सजदा करते है;



build vocabulary

In [169]:
from collections import Counter

def build_vocab(sentences, max_size=3000):

    counter = Counter()

    for sentence in sentences:

        tokens = sentence.split()

        counter.update(tokens)

    vocab = {
        "<PAD>": 0,
        "<SOS>": 1,
        "<EOS>": 2,
        "<UNK>": 3
    }

    most_common = counter.most_common(max_size)

    for word, _ in most_common:

        vocab[word] = len(vocab)

    return vocab

In [170]:
en_vocab = build_vocab(train_en, max_size=3000)
hi_vocab = build_vocab(train_hi, max_size=3000)

In [171]:
train_en = train_en[:5000]
train_hi = train_hi[:5000]

numericalise sentences

In [172]:
def numericalize(sentence, vocab):

    tokens = sentence.split()

    ids = []

    for token in tokens:

        ids.append(
            vocab.get(token, vocab["<UNK>"])
        )

    return ids

add special tokens

In [173]:
def prepare_target(sentence, vocab):

    tokens = sentence.split()

    ids = [vocab["<SOS>"]]

    for token in tokens:

        ids.append(
            vocab.get(token, vocab["<UNK>"])
        )

    ids.append(vocab["<EOS>"])

    return ids

convert training data

In [174]:
train_src = [
    numericalize(sentence, en_vocab)
    for sentence in train_en
]

train_tgt = [
    prepare_target(sentence, hi_vocab)
    for sentence in train_hi
]

check output

In [175]:
print(train_en[0])
print(train_src[0])

print()

print(train_hi[0])
print(train_tgt[0])

Other, Private Use
[3, 3, 460]

अन्य, निज़ी उपयोग
[1, 3, 3, 438, 2]


install and import pytorch

In [176]:
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

import random

create dataset class

In [177]:
class TranslationDataset(Dataset):

    def __init__(self, src_data, tgt_data):

        self.src_data = src_data
        self.tgt_data = tgt_data

    def __len__(self):

        return len(self.src_data)

    def __getitem__(self, idx):

        src = torch.tensor(self.src_data[idx])

        tgt = torch.tensor(self.tgt_data[idx])

        return src, tgt

create collate function, and it automatically does padding

In [178]:
PAD_IDX = 0

def collate_fn(batch):

    src_batch = []
    tgt_batch = []

    for src, tgt in batch:

        src_batch.append(src)
        tgt_batch.append(tgt)

    src_batch = pad_sequence(
        src_batch,
        batch_first=True,
        padding_value=PAD_IDX
    )

    tgt_batch = pad_sequence(
        tgt_batch,
        batch_first=True,
        padding_value=PAD_IDX
    )

    return src_batch, tgt_batch

create dataset object

In [179]:
train_dataset = TranslationDataset(
    train_src,
    train_tgt
)

create dataloader

In [180]:
train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=collate_fn
)

test dataloader

In [181]:
src_batch, tgt_batch = next(iter(train_loader))

print(src_batch.shape)
print(tgt_batch.shape)

torch.Size([4, 48])
torch.Size([4, 65])


device (gpu)

In [182]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

cuda


build encoder

In [183]:
class Encoder(nn.Module):

    def __init__(self,
                 input_dim,
                 emb_dim,
                 hidden_dim):

        super().__init__()

        self.embedding = nn.Embedding(
            input_dim,
            emb_dim
        )

        self.lstm = nn.LSTM(
            emb_dim,
            hidden_dim,
            batch_first=True
        )

    def forward(self, src):

        embedded = self.embedding(src)

        outputs, (hidden, cell) = self.lstm(embedded)

        return hidden, cell

build decoder

In [184]:
class Decoder(nn.Module):

    def __init__(self,
                 output_dim,
                 emb_dim,
                 hidden_dim):

        super().__init__()

        self.embedding = nn.Embedding(
            output_dim,
            emb_dim
        )

        self.lstm = nn.LSTM(
            emb_dim,
            hidden_dim,
            batch_first=True
        )

        self.fc = nn.Linear(
            hidden_dim,
            output_dim
        )

    def forward(self,
                x,
                hidden,
                cell):

        x = x.unsqueeze(1)

        embedded = self.embedding(x)

        outputs, (hidden, cell) = self.lstm(
            embedded,
            (hidden, cell)
        )

        prediction = self.fc(
            outputs.squeeze(1)
        )

        return prediction, hidden, cell

build seq2seq model

In [185]:
class Seq2Seq(nn.Module):

    def __init__(self,
                 encoder,
                 decoder,
                 device):

        super().__init__()

        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self,
                src,
                tgt,
                teacher_forcing_ratio=0.5):

        batch_size = tgt.shape[0]

        tgt_len = tgt.shape[1]

        tgt_vocab_size = self.decoder.fc.out_features

        outputs = torch.zeros(
            batch_size,
            tgt_len,
            tgt_vocab_size
        ).to(self.device)

        hidden, cell = self.encoder(src)

        x = tgt[:, 0]

        for t in range(1, tgt_len):

            output, hidden, cell = self.decoder(
                x,
                hidden,
                cell
            )

            outputs[:, t] = output

            best_guess = output.argmax(1)

            x = tgt[:, t] if random.random() < teacher_forcing_ratio else best_guess

        return outputs

create model

In [186]:
INPUT_DIM = len(en_vocab)
OUTPUT_DIM = len(hi_vocab)

EMB_DIM = 64
HIDDEN_DIM = 128

encoder = Encoder(
    INPUT_DIM,
    EMB_DIM,
    HIDDEN_DIM
)

decoder = Decoder(
    OUTPUT_DIM,
    EMB_DIM,
    HIDDEN_DIM
)

model = Seq2Seq(
    encoder,
    decoder,
    device
).to(device)

loss + optimiser

In [187]:
optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

criterion = nn.CrossEntropyLoss(
    ignore_index=PAD_IDX
)

In [188]:
best_loss = float('inf')

patience = 3

counter = 0

training loop

In [189]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [191]:
num_epochs = 25
torch.cuda.empty_cache()

for epoch in range(num_epochs):

    model.train()

    epoch_loss = 0

    for src, tgt in train_loader:

        src = src.to(device)
        tgt = tgt.to(device)

        optimizer.zero_grad()

        output = model(src, tgt)

        output_dim = output.shape[-1]

        output = output[:, 1:].reshape(-1, output_dim)

        tgt = tgt[:, 1:].reshape(-1)

        loss = criterion(output, tgt)

        loss.backward()

        optimizer.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)

    print(f"Epoch {epoch+1}")
    print(f"Loss: {avg_loss}")
    if avg_loss < best_loss:
        best_loss = avg_loss
        counter = 0
        torch.save(model.state_dict(), "best_model.pth")
    else:
        counter += 1
    if counter >= patience:
         print("Early stopping triggered")
         break

Epoch 1
Loss: 3.954951204919815
Epoch 2
Loss: 3.8919416624069214
Epoch 3
Loss: 3.8118508437156677
Epoch 4
Loss: 3.742002868294716
Epoch 5
Loss: 3.657481570148468
Epoch 6
Loss: 3.5818085311412813
Epoch 7
Loss: 3.517863627386093
Epoch 8
Loss: 3.466350014781952
Epoch 9
Loss: 3.413565155696869
Epoch 10
Loss: 3.3344913108825684
Epoch 11
Loss: 3.2677716829776764
Epoch 12
Loss: 3.2306577458143235
Epoch 13
Loss: 3.170502976989746
Epoch 14
Loss: 3.1468957638263704
Epoch 15
Loss: 3.0768838174819946
Epoch 16
Loss: 3.0206998959541322
Epoch 17
Loss: 2.9529606701612474
Epoch 18
Loss: 2.8975100793361666
Epoch 19
Loss: 2.880448442053795
Epoch 20
Loss: 2.8237521450042724
Epoch 21
Loss: 2.747547168684006
Epoch 22
Loss: 2.7424343693971633
Epoch 23
Loss: 2.686563641893864
Epoch 24
Loss: 2.6637282366514206
Epoch 25
Loss: 2.583497105312347
